In [ ]:
%load_ext autoreload
%autoreload 2

# Elo Rating System — Benchmarking Trained Models

This notebook loads checkpoints from training runs and rates them using the Elo system.
We can compare checkpoints across iterations, training runs, and MCTS configurations.

In [ ]:
from pymcts.core.players import MCTSPlayer, RandomPlayer, GreedyMCTSPlayer
from pymcts.core.config import MCTSConfig
from pymcts.games.bridgit.game import BridgitGame
from pymcts.games.bridgit.config import BoardConfig
from pymcts.elo import (
    RatedPlayer, run_tournament, compute_elo_ratings,
    TournamentConfig, TournamentResult, MatchResult,
)

## 1. Setup — Load checkpoints as players

Load checkpoints from a training run at different iterations.
Each checkpoint + MCTS config = a distinct rated player.

In [ ]:
# Configure which training run and iterations to benchmark
RUN_DIR = "trainings/run_2026-03-29_100200"
ITERATIONS = [1, 10, 20, 30, 40, 50, 60, 70, 80, 89]
BOARD_SIZE = 6

board_config = BoardConfig(size=BOARD_SIZE)
game_factory = lambda: BridgitGame(board_config)

# Load checkpoints as players
players = [RatedPlayer.from_random()]  # anchor at 1000

for it in ITERATIONS:
    iter_path = f"{RUN_DIR}/iteration_{it:03d}"
    mcts_player = MCTSPlayer.from_training_iteration(iter_path, temperature=0.0)
    players.append(RatedPlayer.from_mcts_player(mcts_player))

print(f"Loaded {len(players)} players:")
for p in players:
    print(f"  - {p.name}")

## 2. Run Tournament

Swiss-style tournament: players are matched by rating, no repeat matchups.
40 games per matchup with player swapping to account for first-move advantage.

In [ ]:
config = TournamentConfig(
    games_per_matchup=40,
    swap_players=True,
    num_rounds=None,  # auto: max(3, ceil(log2(num_players)))
    batch_size=8,
)

result = run_tournament(
    players=players,
    game_factory=game_factory,
    config=config,
)

print(f"Tournament complete: {len(result.match_results)} matchups played")
print(f"Anchor: {result.anchor_player} = {result.anchor_rating}")
print()
print("Ratings:")
for r in result.ratings:
    print(f"  {r.name:>20s}  {r.rating:7.1f}  ({r.games_played} games)")

## 3. Visualize Elo progression

Plot how Elo rating evolves across training iterations.

In [ ]:
import plotly.graph_objects as go

by_name = {r.name: r.rating for r in result.ratings}

# Extract iteration numbers and ratings (skip "random")
iter_ratings = []
for it in ITERATIONS:
    name = f"iteration_{it:03d}"
    if name in by_name:
        iter_ratings.append((it, by_name[name]))

iters, ratings = zip(*iter_ratings)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(iters), y=list(ratings),
    mode="lines+markers",
    name="Checkpoint Elo",
    marker=dict(size=8),
))
fig.add_hline(y=1000, line_dash="dash", line_color="gray",
              annotation_text="RandomPlayer (1000)")
fig.update_layout(
    title="Elo Rating vs Training Iteration",
    xaxis_title="Training Iteration",
    yaxis_title="Elo Rating",
    template="plotly_white",
)
fig.show()

## 4. Match results heatmap

Visualize win rates between all pairs of players.

In [ ]:
import numpy as np

# Build win rate matrix from match results
player_names = [r.name for r in result.ratings]  # sorted by rating desc
name_to_idx = {n: i for i, n in enumerate(player_names)}
n = len(player_names)
win_matrix = np.full((n, n), np.nan)

for m in result.match_results:
    if m.player_a in name_to_idx and m.player_b in name_to_idx:
        i, j = name_to_idx[m.player_a], name_to_idx[m.player_b]
        total = m.wins_a + m.wins_b + m.draws
        if total > 0:
            win_matrix[i][j] = m.wins_a / total
            win_matrix[j][i] = m.wins_b / total

fig = go.Figure(data=go.Heatmap(
    z=win_matrix,
    x=player_names,
    y=player_names,
    colorscale="RdYlGn",
    zmin=0, zmax=1,
    text=np.where(np.isnan(win_matrix), "", np.round(win_matrix, 2).astype(str)),
    texttemplate="%{text}",
    colorbar=dict(title="Win Rate"),
))
fig.update_layout(
    title="Win Rate Matrix (row vs column)",
    xaxis_title="Opponent",
    yaxis_title="Player",
    template="plotly_white",
    width=800, height=700,
)
fig.show()

## 5. MCTS parameter impact on Elo

Test how the number of MCTS simulations affects a checkpoint's Elo.
Same neural net, different search budgets.

In [ ]:
# Pick a strong checkpoint and vary MCTS simulations
CHECKPOINT = f"{RUN_DIR}/iteration_089"
SIM_COUNTS = [10, 25, 50, 100, 200]

from pymcts.games.bridgit.neural_net import BridgitNet
from pymcts.games.bridgit.config import NeuralNetConfig

sim_players = [RatedPlayer.from_random()]

for sims in SIM_COUNTS:
    player = MCTSPlayer.from_training_iteration(
        CHECKPOINT,
        mcts_config=MCTSConfig(num_simulations=sims, num_parallel_leaves=min(sims, 10)),
        temperature=0.0,
        name=f"iter89_{sims}sims",
    )
    sim_players.append(RatedPlayer.from_mcts_player(player))

print(f"Players for MCTS parameter study:")
for p in sim_players:
    print(f"  - {p.name}")

In [ ]:
sim_result = run_tournament(
    players=sim_players,
    game_factory=game_factory,
    config=TournamentConfig(games_per_matchup=40, swap_players=True, batch_size=8),
)

print("MCTS simulations vs Elo:")
for r in sim_result.ratings:
    print(f"  {r.name:>20s}  {r.rating:7.1f}")

In [ ]:
sim_by_name = {r.name: r.rating for r in sim_result.ratings}

sim_elos = [(sims, sim_by_name[f"iter89_{sims}sims"]) for sims in SIM_COUNTS if f"iter89_{sims}sims" in sim_by_name]
s, e = zip(*sim_elos)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(s), y=list(e),
    mode="lines+markers",
    name="iter_089",
    marker=dict(size=8),
))
fig.add_hline(y=1000, line_dash="dash", line_color="gray",
              annotation_text="RandomPlayer (1000)")
fig.update_layout(
    title="Elo vs MCTS Simulations (same checkpoint)",
    xaxis_title="Number of MCTS Simulations",
    yaxis_title="Elo Rating",
    xaxis_type="log",
    template="plotly_white",
)
fig.show()

## 6. Save and load results

Tournament results persist as JSON. You can load them later, add new players, and recompute.

In [ ]:
from pathlib import Path

# Save
save_path = Path("elo_results.json")
save_path.write_text(result.model_dump_json(indent=2))
print(f"Saved to {save_path} ({save_path.stat().st_size / 1024:.1f} KB)")

# Load
loaded = TournamentResult.model_validate_json(save_path.read_text())
print(f"Loaded {len(loaded.ratings)} ratings, {len(loaded.match_results)} match results")

# Recompute ratings from loaded match history (proves order-independence)
recomputed = compute_elo_ratings(loaded.match_results, anchor_player="random")
for orig, recomp in zip(result.ratings, recomputed):
    assert abs(orig.rating - recomp.rating) < 0.01, f"Mismatch for {orig.name}"
print("Recomputed ratings match original — ML Elo is order-independent.")

## 7. Add new players to an existing tournament

Load previous results and run additional rounds with new players.
Already-played matchups are skipped automatically.

In [ ]:
# Add a new player (e.g., iteration 5 and 15 that weren't in the original)
new_iterations = [5, 15]
expanded_players = list(players)  # copy original list

for it in new_iterations:
    iter_path = f"{RUN_DIR}/iteration_{it:03d}"
    mcts_player = MCTSPlayer.from_training_iteration(iter_path, temperature=0.0)
    expanded_players.append(RatedPlayer.from_mcts_player(mcts_player))

# Run tournament with previous results — skips already-played matchups
expanded_result = run_tournament(
    players=expanded_players,
    game_factory=game_factory,
    config=config,
    previous_results=loaded,
)

print(f"Expanded tournament: {len(expanded_result.match_results)} total matchups")
print(f"(was {len(loaded.match_results)}, added {len(expanded_result.match_results) - len(loaded.match_results)} new)")
print()
print("Updated ratings:")
for r in expanded_result.ratings:
    print(f"  {r.name:>20s}  {r.rating:7.1f}")